# Cinemática Radar: Cálculo Analítico de CPA y TCPA

El CPA (Closest Point of Approach) es la distancia mínima a la que pasará un buque cercano, y el TCPA es el tiempo que falta para que eso ocurra. Es vital para prevenir colisiones en mar abierto o con mala visibilidad. 

Para Capitán de Yate (CY), se resuelve vectorialmente: La Velocidad Relativa ($V_R$) del contacto respecto a nosotros dicta su Movimiento Relativo (MR).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def polar_to_cartesian(speed, course_deg):
    # Norte es 0 grados (eje Y positivo), Este es 90 grados (eje X positivo)
    rad = np.radians(90 - course_deg)
    x = speed * np.cos(rad)
    y = speed * np.sin(rad)
    return np.array([x, y])

def cartesian_to_polar(x, y):
    speed = np.sqrt(x**2 + y**2)
    course_deg = (90 - np.degrees(np.arctan2(y, x))) % 360
    return speed, course_deg

# Datos iniciales
mi_rumbo = 45 # grados
mi_vel = 12 # nudos

# Detección inicial del contacto (t=0)
demora_contacto = 110 # grados verdaderos
distancia_contacto = 8 # millas

# Curso verdadero del contacto
rumbo_contacto = 340 # grados
vel_contacto = 18 # nudos

# 1. Convertir a vectores
V_mio = polar_to_cartesian(mi_vel, mi_rumbo)
V_suyo = polar_to_cartesian(vel_contacto, rumbo_contacto)

# 2. Vector Velocidad Relativa (Vr = V_suyo - V_mio)
V_relativa = V_suyo - V_mio
vel_rel, rumbo_rel = cartesian_to_polar(V_relativa[0], V_relativa[1])

print(f"Mi Vector (Vx, Vy): {V_mio}")
print(f"Vector Contacto (Vx, Vy): {V_suyo}")
print(f"\nVelocidad Relativa del Contacto: {vel_rel:.2f} nudos")
print(f"Rumbo Relativo (Movimiento Aparente): {rumbo_rel:.1f}°")


### Cálculo de TCPA y CPA

La distancia relativa d se proyecta matemáticamente para hallar cuándo será perpendicular a la línea de avance relativo.

In [ ]:
# Posición inicial relativa (Nosotros estamos en el origen 0,0)
pos_rel_inicial = polar_to_cartesian(distancia_contacto, demora_contacto)

# Cálculo matemático puro (Producto punto)
# TCPA = -(Posición · V_relativa) / |V_relativa|^2
dot_product = np.dot(pos_rel_inicial, V_relativa)
tcpa_hours = -dot_product / (vel_rel**2)

if tcpa_hours < 0:
    print("El TCPA es negativo. El barco ya pasó su punto más cercano.")
else:
    # Posición en CPA = Pos_inicial + V_relativa * TCPA
    pos_cpa = pos_rel_inicial + V_relativa * tcpa_hours
    cpa_distance = np.linalg.norm(pos_cpa)
    
    print(f"Tiempo hasta CPA (TCPA): {tcpa_hours*60:.1f} minutos")
    print(f"Distancia en el CPA: {cpa_distance:.2f} millas")
    
    if cpa_distance < 2.0:
        print("🚨 ¡PELIGRO DE ABORDAJE! CPA menor a 2 millas.")
    else:
        print("✅ Cruce seguro.")
        
# Gráfico Vectorial
plt.figure(figsize=(8,8))
plt.grid(True)
plt.axhline(0, color='black',linewidth=0.5)
plt.axvline(0, color='black',linewidth=0.5)

# Trazar nuestro barco
plt.plot(0, 0, 'go', markersize=10, label="Nuestro Barco (Origen)")

# Trazar contacto
plt.plot(pos_rel_inicial[0], pos_rel_inicial[1], 'ro', label="Contacto (t=0)")

# Línea de Movimiento Relativo (LMR)
tiempo_proy = np.linspace(0, max(0.1, tcpa_hours*1.5), 50)
x_tray = pos_rel_inicial[0] + V_relativa[0] * tiempo_proy
y_tray = pos_rel_inicial[1] + V_relativa[1] * tiempo_proy
plt.plot(x_tray, y_tray, 'r--', label="Línea Mov. Relativo")

# Trazar CPA
if tcpa_hours >= 0:
    plt.plot(pos_cpa[0], pos_cpa[1], 'bx', markersize=12, label="Punto CPA")
    plt.plot([0, pos_cpa[0]], [0, pos_cpa[1]], 'b:', label=f"CPA: {cpa_distance:.2f} nm")

plt.xlim(-10, 10)
plt.ylim(-10, 10)
plt.xlabel("Este/Oeste (Millas)")
plt.ylabel("Norte/Sur (Millas)")
plt.title("Cinemática Radar (Plotting Vectorial)")
plt.legend()
plt.show()
